In [38]:
import pandas as pd
import numpy as np
import time
import json
import requests
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm.auto import tqdm
from math import radians, cos, sin, asin, sqrt

In [39]:
drivers = pd.read_csv("drivers.csv")
rides = pd.read_csv("ncr_ride_bookings.csv")
ratings = pd.read_csv("ratings.csv")
retail = pd.read_excel("Online Retail.xlsx")

In [40]:
print("drivers:", drivers.shape)
print(drivers.columns.tolist())
drivers.head()

drivers: (300, 6)
['driver_id', 'name', 'vehicle_id', 'rating', 'total_rides', 'available']


,driver_id,name,vehicle_id,rating,total_rides,available
0,1,Virginia Johnson,284,3.44,30,True
1,2,Daniel Maddox,18,2.59,61,False
2,3,Patricia Salazar,298,4.93,75,True
3,4,Cameron Hinton DDS,49,2.59,12,True
4,5,Jennifer Ayala,138,4.69,71,True


In [41]:
print("\nrides:", rides.shape)
print(rides.columns.tolist())
rides.head()


rides: (150000, 21)
['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID', 'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT', 'Avg CTAT', 'Cancelled Rides by Customer', 'Reason for cancelling by Customer', 'Cancelled Rides by Driver', 'Driver Cancellation Reason', 'Incomplete Rides', 'Incomplete Rides Reason', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating', 'Payment Method']


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [42]:
rides.columns = [c.strip().replace(" ", "_").lower() for c in rides.columns]
rides.head()

,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,...,reason_for_cancelling_by_customer,cancelled_rides_by_driver,driver_cancellation_reason,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [43]:
print("\nratings:", ratings.shape)
print(ratings.columns.tolist())
ratings.head()


ratings: (50000, 6)
['rating_id', 'ride_id', 'user_id', 'rating_value', 'comments', 'rating_date']


,rating_id,ride_id,user_id,rating_value,comments,rating_date
0,1,1,3046,3,Million onto whom everything.,2024-07-18 23:43:41
1,2,2,3352,5,NaN,2024-04-15 07:00:17
2,3,3,658,4,Tonight another goal lead.,2024-04-21 20:25:52
3,4,4,7893,3,NaN,2024-03-22 21:24:44
4,5,5,2466,4,NaN,2024-01-11 21:13:34


In [44]:
print("\nretail:", retail.shape)
print(retail.columns.tolist())
retail.head()


retail: (541909, 8)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [45]:
retail.columns = [c.strip().replace(" ", "_").lower() for c in retail.columns]
retail.head()


,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [46]:
if 'booking_status' in rides.columns:
    rides = rides[rides['booking_status'].str.lower() == 'completed']
else:
    print("No 'booking_status' column found. Keep all rides for now.")

In [47]:
rides_subset = rides[[
    col for col in ['booking_id','customer_id','pickup_location','drop_location',
                    'ride_distance','booking_value','vehicle_type','driver_id',
                    'driver_ratings','customer_rating','date','time'] if col in rides.columns
]].copy()

drivers_subset = drivers[[
    col for col in ['driver_id','rating','total_rides','vehicle_id','available'] if col in drivers.columns
]].copy()

ratings_subset = ratings[[
    col for col in ['ride_id','user_id','rating_value','comments'] if col in ratings.columns
]].copy()

retail_subset = retail[[
    col for col in ['customerid','quantity','unitprice'] if col in retail.columns
]].copy()

print("rides_subset:", rides_subset.shape)
print("drivers_subset:", drivers_subset.shape)
print("ratings_subset:", ratings_subset.shape)
print("retail_subset:", retail_subset.shape)

rides_subset: (93000, 11)
drivers_subset: (300, 5)
ratings_subset: (50000, 4)
retail_subset: (541909, 3)


In [48]:
ratings_subset.isnull().sum()

ride_id             0
user_id             0
rating_value        0
comments        25103
dtype: int64

In [49]:
ratings_subset.loc[:, 'comments'] = ratings_subset['comments'].fillna('No comment')
print(ratings_subset.isnull().sum())

ride_id         0
user_id         0
rating_value    0
comments        0
dtype: int64


In [50]:
retail_subset = retail_subset.dropna(subset=['customerid'])

retail_subset['customerid'] = retail_subset['customerid'].astype(int)

print("Remaining nulls:", retail_subset['customerid'].isnull().sum())
print("Unique customers:", retail_subset['customerid'].nunique())


Remaining nulls: 0
Unique customers: 4372


In [51]:
geolocator = Nominatim(user_agent="community_marketplace_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, error_wait_seconds=5)

locs = pd.Series(pd.concat([
    rides_subset['pickup_location'].dropna().astype(str),
    rides_subset['drop_location'].dropna().astype(str)
]).unique())

In [52]:
import os
cache_file = "location_cache.csv"

if os.path.exists(cache_file):
    cache_df = pd.read_csv(cache_file, index_col=0)
    print("Loaded cached locations:", len(cache_df))
else:
    cache_df = pd.DataFrame(columns=["lat","lon"])
    cache_df.index.name = "location"

geolocator = Nominatim(user_agent="community_marketplace_geocoder", timeout=10)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    error_wait_seconds=5,
    max_retries=2,
    swallow_exceptions=True
)

new_rows = []
for loc in tqdm(locs, desc="Geocoding unique locations"):
    if loc in cache_df.index:
        continue
    try:
        res = geocode(loc, timeout=10)
        if res:
            new_rows.append({"location": loc, "lat": res.latitude, "lon": res.longitude})
        else:
            new_rows.append({"location": loc, "lat": np.nan, "lon": np.nan})
    except Exception as e:
        print("Error geocoding", loc, repr(e))
        new_rows.append({"location": loc, "lat": np.nan, "lon": np.nan})

if new_rows:
    new_df = pd.DataFrame(new_rows).set_index("location")
    new_df = new_df.dropna(subset=["lat","lon"], how="all")
    if not new_df.empty:
        cache_df = pd.concat([cache_df, new_df[["lat","lon"]]])
        cache_df.to_csv(cache_file)

rides_subset.loc[:, 'pickup_lat'] = rides_subset['pickup_location'].map(lambda x: cache_df.loc[x, 'lat'] if x in cache_df.index else np.nan)
rides_subset.loc[:, 'pickup_lon'] = rides_subset['pickup_location'].map(lambda x: cache_df.loc[x, 'lon'] if x in cache_df.index else np.nan)
rides_subset.loc[:, 'drop_lat'] = rides_subset['drop_location'].map(lambda x: cache_df.loc[x, 'lat'] if x in cache_df.index else np.nan)
rides_subset.loc[:, 'drop_lon'] = rides_subset['drop_location'].map(lambda x: cache_df.loc[x, 'lon'] if x in cache_df.index else np.nan)

print("Geocoded rides (pickup lat null count):", rides_subset['pickup_lat'].isnull().sum())

Loaded cached locations: 176


Geocoding unique locations: 100%|██████████| 176/176 [00:00<?, ?it/s]


Geocoded rides (pickup lat null count): 1018


In [54]:
def haversine(lon1, lat1, lon2, lat2):
    if any(pd.isnull([lon1, lat1, lon2, lat2])):
        return np.nan
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    km = 6371 * c
    return km

rides_subset['distance_km_actual'] = rides_subset.apply(
    lambda r: haversine(r['pickup_lon'], r['pickup_lat'], r['drop_lon'], r['drop_lat']), axis=1
)

if 'ride_distance' in rides_subset.columns:
    rides_subset['distance_km'] = rides_subset['distance_km_actual'].fillna(rides_subset['ride_distance'])
else:
    rides_subset['distance_km'] = rides_subset['distance_km_actual']

print("distance_km nulls:", rides_subset['distance_km'].isnull().sum())
rides_subset[['distance_km','distance_km_actual','ride_distance']].head()


distance_km nulls: 0


,distance_km,distance_km_actual,ride_distance
2,20.873619,20.873619,13.58
3,7.594143,7.594143,34.02
4,14.368867,14.368867,48.21
5,653.907900,653.907900,4.85
6,878.042032,878.042032,41.24


In [55]:
OWM_API_KEY = '165bf9677bdb53cc97f2cbaa5914025f'

# Prepare a unique list of (lat, lon, date) combinations to query to reduce calls
# If your rides have 'date' or timestamp column, adapt accordingly; otherwise use just lat/lon
rides_subset['date_only'] = rides_subset.get('date', pd.NaT).astype(str)
query_points = rides_subset[['pickup_lat','pickup_lon','date_only']].drop_duplicates()

# Cache file
weather_cache_file = "weather_cache.json"
if os.path.exists(weather_cache_file):
    with open(weather_cache_file, "r") as f:
        weather_cache = json.load(f)
else:
    weather_cache = {}

def fetch_weather(lat, lon):
    key = f"{lat:.5f}_{lon:.5f}"
    if key in weather_cache:
        return weather_cache[key]
    # Using Current Weather API (for historical you'd need paid plan); here we use current as proxy
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={OWM_API_KEY}&units=metric"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            entry = {
                "temp": data.get("main",{}).get("temp"),
                "humidity": data.get("main",{}).get("humidity"),
                "weather": data.get("weather",[{}])[0].get("main")
            }
        else:
            entry = {"temp": None, "humidity": None, "weather": None}
    except Exception as e:
        print("Weather fetch error", e)
        entry = {"temp": None, "humidity": None, "weather": None}
    weather_cache[key] = entry
    # Save incrementally
    with open(weather_cache_file, "w") as f:
        json.dump(weather_cache, f)
    time.sleep(1)  # be polite
    return entry

# Apply (example limited to first 500 unique points to avoid long runs)
for idx, row in tqdm(query_points.head(500).iterrows(), total=min(500, len(query_points))):
    if pd.isnull(row['pickup_lat']) or pd.isnull(row['pickup_lon']):
        continue
    _ = fetch_weather(float(row['pickup_lat']), float(row['pickup_lon']))

# Map weather back to rides_subset
def lookup_weather(lat, lon):
    if pd.isnull(lat) or pd.isnull(lon):
        return pd.Series({"temp":np.nan,"humidity":np.nan,"weather":None})
    key = f"{float(lat):.5f}_{float(lon):.5f}"
    val = weather_cache.get(key, {"temp":None,"humidity":None,"weather":None})
    return pd.Series({"temp": val["temp"], "humidity": val["humidity"], "weather": val["weather"]})

weather_cols = rides_subset.apply(lambda r: lookup_weather(r['pickup_lat'], r['pickup_lon']), axis=1)
rides_subset = pd.concat([rides_subset, weather_cols], axis=1)

print("Weather columns added. Temp nulls:", rides_subset['temp'].isnull().sum())


100%|██████████| 500/500 [03:34<00:00,  2.33it/s]


Weather columns added. Temp nulls: 7296


In [56]:
traffic_zones = {
    "Connaught Place": 0.95,
    "Gurgaon": 0.85,
    "Noida": 0.75,
    "South Delhi": 0.7,
    "Janakpuri": 0.6,
    "Rohini": 0.55,
    "Dwarka": 0.5,
    "Ghitorni": 0.45,
}

def get_zone_index(loc):
    if pd.isnull(loc):
        return np.nan
    for k, v in traffic_zones.items():
        if k.lower() in loc.lower():
            return v
    # else random plausible traffic index
    return float(np.round(np.random.uniform(0.35, 0.85), 2))

rides_subset['traffic_index'] = rides_subset['pickup_location'].apply(get_zone_index)
print("traffic_index distribution:")
print(rides_subset['traffic_index'].describe())


traffic_index distribution:
count    93000.000000
mean         0.611233
std          0.148778
min          0.350000
25%          0.490000
50%          0.610000
75%          0.750000
max          0.950000
Name: traffic_index, dtype: float64


In [61]:
print(rides_subset.columns.tolist())


['booking_id', 'customer_id', 'pickup_location', 'drop_location', 'ride_distance', 'booking_value', 'vehicle_type', 'driver_ratings', 'customer_rating', 'date', 'time', 'pickup_lat', 'pickup_lon', 'drop_lat', 'drop_lon', 'distance_km_actual', 'distance_km', 'date_only', 'temp', 'humidity', 'weather', 'traffic_index', 'service_category', 'skill_match_score']


In [62]:
if 'drivers' not in locals():
    drivers = pd.read_csv("drivers.csv")

all_driver_ids = drivers['driver_id'].tolist()
num_rides = len(rides_subset)

np.random.seed(42)
rides_subset['driver_id'] = np.random.choice(all_driver_ids, size=num_rides)

print("Synthetic driver_id column added.")
print(rides_subset[['pickup_location','vehicle_type','driver_id']].head())


Synthetic driver_id column added.
       pickup_location   vehicle_type  driver_id
2              Khandsa           Auto        103
3  Central Secretariat  Premier Sedan        271
4     Ghitorni Village           Bike        107
5                AIIMS           Auto         72
6             Vaishali        Go Mini        189


In [63]:
# 1. Define possible skill categories
skills = [
    "cleaning", "plumbing", "electrical", "carpentry",
    "painting", "general_maintenance", "moving_help"
]

# 2. Ensure every driver in rides_subset has a skill record
np.random.seed(42)
existing_driver_ids = driver_skills['driver_id'].tolist() if 'driver_skills' in locals() else []
all_driver_ids = pd.Series(rides_subset['driver_id'].dropna().unique()).astype(int).tolist()

missing_ids = [i for i in all_driver_ids if i not in existing_driver_ids]

new_rows = []
for did in missing_ids:
    primary = np.random.choice(skills)
    secondary = np.random.choice([s for s in skills if s != primary])
    exp_level = np.random.choice(["novice","intermediate","expert"], p=[0.2,0.5,0.3])
    p_score = round(np.random.uniform(0.75, 0.99), 2)
    s_score = round(p_score * np.random.uniform(0.6,0.9), 2)
    new_rows.append({
        "driver_id": did,
        "skill_primary": primary,
        "skill_secondary": secondary,
        "experience_level": exp_level,
        "primary_score": p_score,
        "secondary_score": s_score
    })

# Combine with existing if any
if 'driver_skills' in locals():
    driver_skills = pd.concat([driver_skills, pd.DataFrame(new_rows)], ignore_index=True)
else:
    driver_skills = pd.DataFrame(new_rows)

print(f"✅ Driver skills table now covers {len(driver_skills)} drivers")

# 3. Ensure every ride has a valid service category
skills_for_fill = skills.copy()
rides_subset['service_category'] = rides_subset.get('service_category')

# Fill missing or absent service_category intelligently
if rides_subset['service_category'].isnull().all() or 'service_category' not in rides_subset.columns:
    # If none exist, create based on vehicle_type or random
    rides_subset['service_category'] = rides_subset.get('vehicle_type', '').apply(
        lambda x: np.random.choice(skills_for_fill)
    )
else:
    rides_subset['service_category'] = rides_subset['service_category'].fillna(
        rides_subset.get('vehicle_type', '').apply(lambda x: np.random.choice(skills_for_fill))
    )

# 4. Compute skill match score for each ride with fallback
def compute_match_score(row):
    did = row.get('driver_id')
    svc = row.get('service_category')
    if pd.isnull(did) or svc is None:
        return np.random.uniform(0.3, 0.7)
    rec = driver_skills.loc[driver_skills['driver_id'] == did]
    if rec.empty:
        return np.random.uniform(0.3, 0.7)
    rec = rec.iloc[0]
    if rec['skill_primary'] == svc:
        return rec['primary_score']
    elif rec['skill_secondary'] == svc:
        return rec['secondary_score']
    else:
        # Partial match fallback
        return round(np.random.uniform(0.3, 0.7), 2)

rides_subset['skill_match_score'] = rides_subset.apply(compute_match_score, axis=1)

# 5. Scale scores by driver experience level (optional realism)
exp_map = {"novice": 0.8, "intermediate": 1.0, "expert": 1.2}
exp_score = driver_skills.set_index('driver_id')['experience_level'].map(exp_map)
rides_subset['skill_match_score'] *= rides_subset['driver_id'].map(exp_score).fillna(1)

# 6. Inspect results
print("Nulls remaining in skill_match_score:", rides_subset['skill_match_score'].isnull().sum())
print(rides_subset[['driver_id','service_category','skill_match_score']].head(10))
print("\nDistribution summary:\n", rides_subset['skill_match_score'].describe())


✅ Driver skills table now covers 1000 drivers
Nulls remaining in skill_match_score: 0
    driver_id     service_category  skill_match_score
2         103             cleaning              0.828
3         271             cleaning              0.330
4         107           electrical              0.304
5          72             cleaning              0.470
6         189           electrical              0.670
7          21             cleaning              0.370
10        103             cleaning              0.552
13        122  general_maintenance              0.840
14        215           electrical              0.540
15         88             cleaning              0.550

Distribution summary:
 count    93000.000000
mean         0.586044
std          0.192948
min          0.240000
25%          0.440000
50%          0.560000
75%          0.696000
max          1.188000
Name: skill_match_score, dtype: float64


In [68]:
np.random.seed(42)

# 20% of customers are repeat users, with repeat rates between 0.4–1.0
repeat_customers = np.random.choice(rides_subset['customer_id'].unique(),
                                    size=int(len(rides_subset['customer_id'].unique()) * 0.2),
                                    replace=False)

rides_subset['customer_repeat_rate'] = rides_subset['customer_id'].apply(
    lambda x: np.random.uniform(0.4, 1.0) if x in repeat_customers else np.random.uniform(0.0, 0.3)
)
print(rides_subset[['customer_id','customer_repeat_rate']].head(10))


     customer_id  customer_repeat_rate
2   "CID9202816"              0.060486
3   "CID2610914"              0.081027
4   "CID9933542"              0.119606
5   "CID4670564"              0.049973
6   "CID6800553"              0.138358
7   "CID8610436"              0.124815
10  "CID6680340"              0.297026
13  "CID2669710"              0.178159
14  "CID1789354"              0.441677
15  "CID5432215"              0.137982


In [70]:
rides_subset['success_flag'] = 1

if 'time' in rides_subset.columns:
    rides_subset['hour'] = pd.to_datetime(
    rides_subset['time'], format="%H:%M", errors='coerce'
).dt.hour
    rides_subset['peak_hour'] = rides_subset['hour'].apply(lambda h: 1 if h in range(7,10) or h in range(17,20) else 0)
else:
    rides_subset['peak_hour'] = 0

rides_subset['best_driver_id'] = rides_subset['driver_id']

final_cols = [
    'booking_id','customer_id','best_driver_id','pickup_location','drop_location',
    'pickup_lat','pickup_lon','drop_lat','drop_lon','distance_km','booking_value',
    'service_category','skill_match_score','driver_avg_rating','customer_repeat_rate',
    'traffic_index','temp','humidity','weather','peak_hour','success_flag'
]
final_cols = [c for c in final_cols if c in rides_subset.columns]

dispatch_enriched = rides_subset[final_cols].copy()
print("Final enriched shape:", dispatch_enriched.shape)
dispatch_enriched.head()

Final enriched shape: (93000, 21)


,booking_id,customer_id,best_driver_id,pickup_location,drop_location,pickup_lat,pickup_lon,drop_lat,drop_lon,distance_km,...,service_category,skill_match_score,driver_avg_rating,customer_repeat_rate,traffic_index,temp,humidity,weather,peak_hour,success_flag
2,"""CNR8494506""","""CID9202816""",103,Khandsa,Malviya Nagar,28.453328,77.019540,28.533920,77.212447,20.873619,...,cleaning,0.828,4.23,0.060486,0.50,29.19,11.0,Clear,0,1
3,"""CNR8906825""","""CID2610914""",271,Central Secretariat,Inderlok,28.615876,77.212345,28.672707,77.169187,7.594143,...,cleaning,0.330,3.11,0.081027,0.81,29.66,9.0,Clear,0,1
4,"""CNR1950162""","""CID9933542""",107,Ghitorni Village,Khan Market,28.493121,77.142487,28.599514,77.225978,14.368867,...,electrical,0.304,1.82,0.119606,0.45,29.16,10.0,Clear,0,1
5,"""CNR4096693""","""CID4670564""",72,AIIMS,Narsinghpur,28.566860,77.207806,22.934556,79.086785,653.907900,...,cleaning,0.470,2.79,0.049973,0.81,29.65,9.0,Clear,0,1
6,"""CNR2002539""","""CID6800553""",189,Vaishali,Punjabi Bagh,25.749170,85.397216,28.672995,77.146124,878.042032,...,electrical,0.670,2.27,0.138358,0.77,30.46,36.0,Clear,0,1


In [ ]:
nulls = dispatch_enriched.isnull().sum().sort_values(ascending=False)
print("Nulls per column:\n", nulls[nulls>0].head(20))

print("\nService category counts:\n", dispatch_enriched['service_category'].value_counts().head())

dispatch_enriched.to_csv("dispatch_enriched.csv", index=False)
print("Saved dispatch_enriched.csv (rows: {})".format(len(dispatch_enriched)))


Nulls per column:
 humidity      7296
weather       7296
temp          7296
drop_lon      1080
drop_lat      1080
pickup_lat    1018
pickup_lon    1018
dtype: int64

Service category counts:
 service_category
cleaning               30685
moving_help            24332
plumbing                7656
painting                7654
general_maintenance     7589
Name: count, dtype: int64
Saved dispatch_enriched.csv (rows: 93000)
